In [1]:
# %%
from booknlp.booknlp import BookNLP
from pathlib import Path
import torch
import os, re
import pandas as pd
import ast
import json

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# BookNLP config
params = {
    "pipeline": "entity,quote,supersense,event,coref",
    "model": "big",
    "device": "cuda",
}

bnlp = BookNLP("en", params)

/home/ham2176/miniconda3/envs/booknlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-02 12:55:15.580362: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/ham2176/miniconda3/envs/booknlp/lib/python3.10/site-packages/booknlp/english/entity_tagger.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


using device cuda
CUDA available: True
GPU: NVIDIA RTX A6000
{'pipeline': 'entity,quote,supersense,event,coref', 'model': 'big', 'device': 'cuda'}
--- startup: 21.898 seconds ---


In [33]:
%pip install openpyxl
import openpyxl
CORPUS_DIR = "../data/manual/smaller_corpus.xlsx"
corpus = pd.read_excel(CORPUS_DIR)
corpus.head()

Note: you may need to restart the kernel to use updated packages.


,Author,Title,Year,Canonicity,PG,Smaller Corpus,Notes
0,Maria Edgeworth,Castle Rackrent,1800,NaN,1424,NaN,NaN
1,Maria Edgeworth,Belinda,1801,2.0,9455,1.0,NaN
2,Matthew Lewis,The Bravo of Venice,1805,3.0,NaN,NaN,NaN
3,Lady Sydney Morgan,The Wild Irish Girl,1806,1.0,54683,NaN,NaN
4,Maria Edgeworth,Ennui,1809,2.0,NaN,NaN,NaN


In [34]:
corpus["Smaller Corpus"].value_counts()

Smaller Corpus
27.0    2
56.0    2
3.0     1
4.0     1
5.0     1
       ..
55.0    1
57.0    1
58.0    1
59.0    1
60.0    1
Name: count, Length: 62, dtype: int64

In [40]:
#filter corpus where "Smaller Corpus" is not none
corpus = corpus[corpus["Smaller Corpus"] > 0].sort_values(by="Smaller Corpus", ascending=True).reset_index(drop=True)
corpus.head()

,Author,Title,Year,Canonicity,PG,Smaller Corpus,Notes
0,Maria Edgeworth,Belinda,1801,2.0,9455,1.0,NaN
1,Jane Austen,Sense and Sensibility,1811,1.0,161,2.0,NaN
2,Jane Austen,Pride and Prejudice,1813,1.0,1342,3.0,NaN
3,Jane Austen,Mansfield Park,1814,1.0,141,4.0,NaN
4,Walter Scott,Waverley,1814,1.0,4966,5.0,NaN


In [42]:
corpus["PG"].value_counts()

PG
9455                          1
161                           1
1342                          1
141                           1
4966                          1
158                           1
5999                          1
6941                          1
105                           1
53685; 53686; 53687; 53688    1
9840                          1
7735                          1
23564                         1
21553                         1
730                           1
16215                         1
24210                         1
3760                          1
821                           1
51649                         1
7265                          1
2153                          1
30486                         1
4275                          1
2505                          1
786                           1
4276                          1
1860                          1
3409                          1
98                            1
507                           1
6688 

In [50]:
corpus.describe()

,Year,Canonicity,Smaller Corpus
count,64.000000,64.000000,64.000000
mean,1854.859375,1.359375,31.812500
std,26.885924,0.545317,18.025445
min,1801.000000,1.000000,1.000000
25%,1836.750000,1.000000,16.750000
50%,1856.000000,1.000000,31.500000
75%,1874.500000,2.000000,47.250000
max,1900.000000,3.000000,62.000000


In [51]:
corpus.to_csv('corpus.csv', index=False)

In [47]:
def load_text(book_id, base_dir="../data/text"):
    path = os.path.join(base_dir, f'PG{book_id}_text.txt')
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return f.read()
    raise FileNotFoundError(f"No text found for PG{book_id}")

text = load_text(1260)
print(text[:1000])  # print the first 500 characters


JANE EYRE
AN AUTOBIOGRAPHY

by Charlotte Brontë

_ILLUSTRATED BY F. H. TOWNSEND_

London
SERVICE & PATON
5 HENRIETTA STREET
1897

_The Illustrations_
_in this Volume are the copyright of_
SERVICE & PATON, _London_

TO
W. M. THACKERAY, ESQ.,

This Work
IS RESPECTFULLY INSCRIBED

BY
THE AUTHOR




PREFACE


A preface to the first edition of “Jane Eyre” being unnecessary, I gave
none: this second edition demands a few words both of acknowledgment
and miscellaneous remark.

My thanks are due in three quarters.

To the Public, for the indulgent ear it has inclined to a plain tale
with few pretensions.

To the Press, for the fair field its honest suffrage has opened to an
obscure aspirant.

To my Publishers, for the aid their tact, their energy, their practical
sense and frank liberality have afforded an unknown and unrecommended
Author.

The Press and the Public are but vague personifications for me, and I
must thank them in vague terms; but my Publishers are definite: so are
certain gener

In [43]:
#how many are missing PG numbers?
corpus[corpus["PG"].isnull()]

,Author,Title,Year,Canonicity,PG,Smaller Corpus,Notes
9,Walter Scott,The Heart of Midlothian,1818,1.0,NaN,10.0,NaN
11,Walter Scott,The Pirate,1821,2.0,NaN,12.0,NaN
19,Frances Trollope,"The Life and Adventures of Michael Armstrong, ...",1840,1.0,NaN,20.0,NaN
22,William Makepeace Thackeray,Vanity Fair 1847,1847,1.0,NaN,23.0,NaN
58,*Ella Hepworth Dixon,The Story of a Modern Woman,1894,1.0,NaN,57.0,NaN
62,Margaret Oliphant,Phoebe Junior,1876,1.0,NaN,61.0,NaN


In [44]:
# go one directory up from notebooks/ into data/
DATA_DIR = os.path.join("..", "data")
META_PATH = os.path.join(DATA_DIR, "metadata.csv")

meta = pd.read_csv(META_PATH)
meta.head()

,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects,type
0,PG10000,The Magna Carta,Anonymous,NaN,NaN,['en'],1083,"{'Magna Carta', 'Constitutional history -- Eng...",Text
1,PG10001,Apocolocyntosis,"Seneca, Lucius Annaeus",NaN,65.0,['en'],2048,"{'Claudius, Emperor of Rome, 10 B.C.-54 A.D. -...",Text
2,PG10002,The House on the Borderland,"Hodgson, William Hope",1877.0,1918.0,['en'],1752,{'Science fiction'},Text
3,PG10003,"My First Years as a Frenchwoman, 1876-1879","Waddington, Mary King",1833.0,1923.0,['en'],260,"{'France -- History -- Third Republic, 1870-19...",Text
4,PG10004,The Warriors,"Lindsay, Anna Robertson Brown",1864.0,1948.0,['en'],205,{'Christianity'},Text


In [45]:
def find_book(title):
    book = meta[meta["title"].str.contains(f"{title}", case=False, na=False)]
    return book["id"], book["title"], book["author"]

book_id, book_title, book_author = find_book("Jane Eyre")
#print results of query sequentially
for i in range(len(book_id)):
    print(f"ID: {book_id.iloc[i]}, Title: {book_title.iloc[i]}, Author: {book_author.iloc[i]}")

ID: PG1260, Title: Jane Eyre: An Autobiography, Author: Brontë, Charlotte
ID: PG16235, Title: Jane Eyre; ou Les mémoires d'une institutrice, Author: Brontë, Charlotte
ID: PG23077, Title: Jane Eyre, Author: Brontë, Charlotte
ID: PG40655, Title: The Key to the Brontë Works: The Key to Charlotte Brontë's 'Wuthering Heights,' 'Jane Eyre,' and her other works., Author: Malham-Dembleby, John
ID: PG47275, Title: Kotiopettajattaren romaani (Jane Eyre), Author: Brontë, Charlotte


In [46]:
book_id, book_title, book_author = find_book("Wilhelm Meister's")
print(f"Book ID: {book_id}, Title: {book_title}, Author: {book_author}")

Book ID: 29380    PG36483
Name: id, dtype: object, Title: 29380    Wilhelm Meister's Apprenticeship and Travels, ...
Name: title, dtype: object, Author: 29380    Goethe, Johann Wolfgang von
Name: author, dtype: object


In [8]:
input_file = Path("/local/nlp/ham2176/gutenberg/data/text/PG1260_text.txt")

output_dir = Path("/local/nlp/ham2176/gutenberg/jane/")
output_dir.mkdir(parents=True, exist_ok=True)

book_id = "jane_eyre"

print(f"Processing full novel: {input_file}")
bnlp.process(str(input_file), str(output_dir), book_id)

print("Finished processing full novel!")

Processing full novel: /local/nlp/ham2176/gutenberg/data/text/PG1260_text.txt
--- spacy: 35.189 seconds ---
--- entities: 135.397 seconds ---
--- quotes: 0.232 seconds ---
--- attribution: 44.662 seconds ---
--- name coref: 0.904 seconds ---
--- coref: 40.775 seconds ---
--- generating character JSON: start ---
--- character JSON: 47.322 seconds ---
--- generating simplified character JSON: start ---
--- simplified character JSON: 40.602 seconds ---
--- generating tagged book: start ---
--- tagged book: 0.654 seconds ---
--- TOTAL (excl. startup): 346.742 seconds ---, 230455 words
Finished processing full novel!
